# Brinjal Disease Detection System
Train an image classification model using TensorFlow, Keras, and EfficientNetB0 transfer learning to detect brinjal (eggplant) diseases (healthy vs little leaf).

In [ ]:
# ======================
# IMPORTS
# ======================
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing import image

print("TensorFlow:", tf.__version__)

# ======================
# CONFIG
# ======================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

BASE_DIR = os.getcwd()   # IMPORTANT FIX
DATA_DIR = os.path.join(BASE_DIR, "dataset")
MODEL_DIR = os.path.join(BASE_DIR, "model")

os.makedirs(MODEL_DIR, exist_ok=True)

print("Dataset path:", DATA_DIR)
print("Exists:", os.path.exists(DATA_DIR))

# ======================
# DATA GENERATOR
# ======================
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print("Classes:", train_gen.class_indices)

# ======================
# SAVE LABELS
# ======================
labels_path = os.path.join(MODEL_DIR, "brinjal_labels.json")

class_labels = {v: k for k, v in train_gen.class_indices.items()}

with open(labels_path, "w") as f:
    json.dump(class_labels, f, indent=4)

# ======================
# MODEL
# ======================
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ======================
# CALLBACKS
# ======================
checkpoint_path = os.path.join(MODEL_DIR, "brinjal_model.keras")

callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# ======================
# TRAINING
# ======================
history = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks_list
)

# ======================
# FINE TUNING
# ======================
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=callbacks_list
)

print("Training complete!")

# ======================
# PREDICTION FUNCTION
# ======================
def predict_disease(img_path):
    model = tf.keras.models.load_model(checkpoint_path)

    with open(labels_path, "r") as f:
        labels = json.load(f)

    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)[0]
    class_idx = np.argmax(preds)
    confidence = float(preds[class_idx])

    return {
        "disease": labels[str(class_idx)],
        "confidence": round(confidence * 100, 2)
    }

# Example:
# print(predict_disease("test.jpg"))

TensorFlow: 2.21.0
Dataset path: F:\krishi nova project ml part\brinjal\dataset
Exists: True
Found 1488 images belonging to 2 classes.
Found 370 images belonging to 2 classes.
Classes: {'brinjal_healthy': 0, 'brinjal_little_leaf': 1}


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)          │ (None, 7, 7, 1280)          │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 1280)                │           5,120 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 2)                   │             258 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,218,917 (16.09 MB)

 Trainable params: 166,786 (651.51 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

Epoch 1/15
12/47 ━━━━━━━━━━━━━━━━━━━━ 2:53 5s/step - accuracy: 0.4229 - loss: 1.0082